# Import Bibliotek

In [39]:
import pandas as pd
# https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

# Wczytanie i przygotowanie danych

In [40]:
ocena_subskrypcji = pd.read_csv('../data/raw/ocena_subskrypcji.csv', sep = ';')
df_model = ocena_subskrypcji.copy()
df_model.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [41]:
del df_model['duration']
df_model.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,1,-1,0,unknown,no


In [42]:
df_model['y'] = df_model['y'].map({'yes': 1, 'no': 0})
df_final = pd.get_dummies(df_model, drop_first = True)

In [43]:
df_final.head()

,age,balance,day,campaign,pdays,previous,y,job_blue-collar,job_entrepreneur,job_housemaid,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
0,58,2143,5,1,-1,0,0,False,False,False,...,False,False,False,True,False,False,False,False,False,True
1,44,29,5,1,-1,0,0,False,False,False,...,False,False,False,True,False,False,False,False,False,True
2,33,2,5,1,-1,0,0,False,True,False,...,False,False,False,True,False,False,False,False,False,True
3,47,1506,5,1,-1,0,0,True,False,False,...,False,False,False,True,False,False,False,False,False,True
4,33,1,5,1,-1,0,0,False,False,False,...,False,False,False,True,False,False,False,False,False,True


* **Usunięcie kolumny duration (zapobieganie wyciekowi danych).**
* **Mapowanie zmiennych kategorycznych (get_dummies).**
* **Mapowanie zmiennej  y na wartości binarne (0/1).**

# Regresja Logistyczna

In [44]:
X = df_final.drop('y', axis = 1)
Y = df_final['y']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 0)

In [45]:
model = LogisticRegression(max_iter = 10000, class_weight='balanced')
model.fit(X_train, Y_train)
y_pred = model.predict(X_test)

In [46]:
metrics.confusion_matrix(Y_test, y_pred)

array([[6173, 1807],
       [ 404,  659]])

In [47]:
print(metrics.classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.77      0.85      7980
           1       0.27      0.62      0.37      1063

    accuracy                           0.76      9043
   macro avg       0.60      0.70      0.61      9043
weighted avg       0.86      0.76      0.79      9043



**Zastosowanie parametru class_weight='balanced' pozwoliło na uzyskanie wysokiej czułości (Recall = 0.62). Oznacza to, że model poprawnie identyfikuje większość klientów decydujących się na subskrypcje.**

# Normalizacja

**Zastosowano StandardScaler, aby ujednolicić skalę zmiennych.**

In [48]:
num_cols = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

In [49]:
model2 = LogisticRegression(max_iter=1000, class_weight='balanced')
model2.fit(X_train_scaled, Y_train)
y_pred2 = model2.predict(X_test_scaled)

In [50]:
metrics.confusion_matrix(Y_test, y_pred2)

array([[6163, 1817],
       [ 402,  661]])

In [51]:
print(metrics.classification_report(Y_test, y_pred2))

              precision    recall  f1-score   support

           0       0.94      0.77      0.85      7980
           1       0.27      0.62      0.37      1063

    accuracy                           0.75      9043
   macro avg       0.60      0.70      0.61      9043
weighted avg       0.86      0.75      0.79      9043



**Wyniki modelu pozostały stabilne, jednak dzięki standaryzacji model stał się bardziej odporny na różnice w skali zmiennych. Dodatkowo umożliwiło to analizę ważności cech na podstawie współczynników modelu.**

# Analiza cech:


In [52]:
wagi_df = pd.DataFrame({
    'Cecha': X.columns,
    'Waga': model2.coef_[0]
})

In [53]:
wagi_df.sort_values(by="Waga", ascending=True)

,Cecha,Waga
26,contact_unknown,-1.287274
30,month_jan,-1.017102
35,month_nov,-0.815126
27,month_aug,-0.780294
31,month_jul,-0.658463
23,housing_yes,-0.465372
34,month_may,-0.464241
29,month_feb,-0.390800
24,loan_yes,-0.387306
8,job_housemaid,-0.375765


In [54]:
wagi_df.sort_values(by="Waga", ascending=False)

,Cecha,Waga
39,poutcome_success,2.355949
33,month_mar,1.245344
36,month_oct,0.883520
37,month_sep,0.800765
28,month_dec,0.755160
10,job_retired,0.461109
13,job_student,0.302003
20,education_tertiary,0.268027
15,job_unemployed,0.211814
21,education_unknown,0.161665


## Wagi Pozytywne
* Sukces w przeszłości (poutcome_success): Klient który wcześniej zdecydował sie na ofertę z dużym prawdopodobieństwem zdecyduje się jeszcze raz.
* Właściwy moment (month_mar, month_oct): Gorące okresy kiedy klienci częściej decydują się na subskrypcje.
* Zatrudnienie (job_retired, job_student): Emeryci i studenci są najlepszą grupą docelową.
* Edukacja wyższa: Edukacja (education_tertiary) sprzyja pozytywnej decyzji.

## Wagi Ujemne
* Błędy w danych (contact_unknown): Brak kanału komunikacji zniechęca klienta.
* Niewłaściwy moment (month_jan, month_nov): Okresy poświąteczne i przedświąteczne to najgorszy czas na wysyłanie ofert.
* Zadłużenie (housing_yes, loan_yes): Kredytobiorcy rzadko decydują się na subskrypcje.